In [1]:
# CSE 5526 - HW3 Q4: Recurrent Neural Network with Recurrent Hidden Units
# -----------------------------------------------------------------------
# Implement the forward and backward passes for a two-layer RNN:
#     - Hidden layer: 4 sigmoid neurons (fully recurrent)
#     - Output layer: 1 LINEAR neuron
#
# Conventions:
#     - Each COLUMN of U and W holds the incoming weights of one hidden neuron.
#       So U has shape (n_input, n_hidden) and W has shape (n_hidden, n_hidden).
#     - V has shape (1, n_hidden); c has shape (1,).
#     - Initial hidden state h(0) = 0.
#     - Error function: E = 1/2 * sum_t (y(t) - d(t))^2
#
# Forward equations:
#     z(t) = U^T x(t) + W^T h(t-1) + b
#     h(t) = sigmoid(z(t))
#     y(t) = V h(t) + c
#
# You must complete:
#     (1) rnn_forward  : compute hidden activations and outputs
#     (2) rnn_backward : compute delta_y(t), delta_h(t), and gradients
#
# DO NOT modify the provided matrices or data.

import numpy as np

np.set_printoptions(precision=4, suppress=True)


def sigmoid(x):
    """Sigmoid activation function."""
    return 1 / (1 + np.exp(-x))


def dsigmoid(y):
    """Derivative of sigmoid when given y = sigmoid(z)."""
    return y * (1 - y)

In [2]:
def rnn_forward(X, U, V, W, b, c):
    """
    Compute the forward pass for all time steps.

    Args:
        X : np.ndarray, shape (n_input, T)
            Input sequence; column t is x(t).
        U : np.ndarray, shape (n_input, n_hidden)   input -> hidden
        V : np.ndarray, shape (1, n_hidden)         hidden -> output
        W : np.ndarray, shape (n_hidden, n_hidden)  hidden -> hidden
        b : np.ndarray, shape (n_hidden,)           hidden biases
        c : np.ndarray, shape (1,)                  output bias

    Returns:
        y : np.ndarray, shape (T,)
            Output values at each time step.
        h : np.ndarray, shape (n_hidden, T+1)
            Hidden activations, with h[:, 0] = 0 (the initial state)
            and h[:, t] = h(t) for t = 1..T.
        z_list : list of length T
            Hidden pre-activations z(t), saved for the backward pass.
    """
    # TODO: Initialize h (with h[:, 0] = 0) and y
    # TODO: Loop over time steps t = 1..T:
    #        - z(t) = U^T x(t) + W^T h(t-1) + b   (note the transposes)
    #        - h(t) = sigmoid(z(t))
    #        - y(t) = V h(t) + c                  (linear output)
    #        - store z(t) in z_list
    n_hidden = U.shape[1]
    T = X.shape[1]
    h = np.zeros((n_hidden, T + 1))
    y = np.zeros(T)
    z_list = []

    for t in range(1, T + 1):
        z = U.T @ X[:, t - 1] + W.T @ h[:, t - 1] + b
        h[:, t] = sigmoid(z)
        y[t - 1] = (V @ h[:, t] + c)[0]
        z_list.append(z)

    return y, h, z_list

In [3]:
def rnn_backward(X, d, y, h, z_list, U, V, W):
    """
    Compute the backward pass using backpropagation through time (BPTT),
    with E = 1/2 * sum_t (y(t) - d(t))^2.

    You must compute:
        delta_y(t) for each time step (output deltas)
        delta_h(t) for each time step (hidden deltas)
        Gradients of E w.r.t. V, U, W, b, and c.

    Args:
        X, d, y, h, z_list : from the forward pass / provided data
        U, V, W : weight matrices

    Returns:
        grads : dict with keys
            'delta_y' : shape (T,)
            'delta_h' : shape (n_hidden, T), column t-1 is delta_h(t)
            'grad_V'  : same shape as V, (1, n_hidden)
            'grad_U'  : same shape as U, (n_input, n_hidden)
            'grad_W'  : same shape as W, (n_hidden, n_hidden)
            'grad_b'  : shape (n_hidden,)
            'grad_c'  : shape (1,)
    """
    # TODO: Initialize arrays for delta_y(t) and delta_h(t)
    # TODO: Compute delta_y(t) = y(t) - d(t)
    # TODO: Loop backward through time (t = T..1):
    #        - propagate error from the output at time t AND from the
    #          hidden layer at time t+1 (through W)
    #        - multiply by the sigmoid derivative to get delta_h(t)
    # TODO: Accumulate gradients for all weights and biases over time.
    #        Make sure each gradient has the same shape as its parameter.
    n_hidden = U.shape[1]
    T = X.shape[1]
    delta_y = y - d
    delta_h = np.zeros((n_hidden, T))

    grad_V = np.zeros_like(V)
    grad_U = np.zeros_like(U)
    grad_W = np.zeros_like(W)
    grad_b = np.zeros(n_hidden)
    grad_c = np.zeros(1)

    future = np.zeros(n_hidden)   # W @ delta_h(t+1); zero at t = T
    for t in range(T, 0, -1):
        delta_h[:, t - 1] = dsigmoid(h[:, t]) * (V[0] * delta_y[t - 1] + future)
        future = W @ delta_h[:, t - 1]

        grad_V += delta_y[t - 1] * h[:, t].reshape(1, -1)
        grad_c += delta_y[t - 1]
        grad_U += np.outer(X[:, t - 1], delta_h[:, t - 1])
        grad_W += np.outer(h[:, t - 1], delta_h[:, t - 1])
        grad_b += delta_h[:, t - 1]

    return {'delta_y': delta_y, 'delta_h': delta_h,
            'grad_V': grad_V, 'grad_U': grad_U, 'grad_W': grad_W,
            'grad_b': grad_b, 'grad_c': grad_c}

In [4]:
def main():
    """Provided constants (DO NOT change)."""
    # Columns of U and W are per-neuron weights.
    U = np.array([
        [-1.1106, -0.0596, -0.4091, -0.0648],
        [-0.6809, -0.6610, -1.2819,  1.0004],
        [ 0.0142,  0.3060, -0.2849, -0.8013]
    ])
    V = np.array([[1.1086, -0.4641, 0.6646, -0.5301]])
    W = np.array([
        [ 0.0918, -0.9585,  0.8270, -0.2248],
        [ 0.2738, -0.3263, -0.1596, -1.3147],
        [-2.1335,  2.4787, -0.8577, -0.0218],
        [ 0.4040,  1.5391,  0.3984, -0.8801]
    ])
    b = np.array([0.0409, 0.1551, -0.3211, 2.1458])
    c = np.array([0.8004])

    X = np.array([
        [-2.5092,  1.9732, -8.8383,  4.1615,  6.6489],
        [ 9.0143, -6.8796,  7.3235, -9.5883, -5.7532],
        [ 4.6399, -6.8801,  2.0223,  9.3982, -6.3635]
    ])
    d = np.array([1.1963, 2.0975, 0.3877, 1.0453, -1.3718])

    # Run once implemented
    try:
        y, h, z_list = rnn_forward(X, U, V, W, b, c)
        grads = rnn_backward(X, d, y, h, z_list, U, V, W)

        print("Q1. Output sequence y(t):\n", y)
        print("\nQ2. Output deltas delta_y(t):\n", grads.get("delta_y"))
        print("\nQ2. Hidden deltas delta_h(t) (column t-1 = time t):\n", grads.get("delta_h"))
        print("\nQ3. Gradients:")
        for k in ["grad_V", "grad_U", "grad_W", "grad_b", "grad_c"]:
            print(f"\n{k}:\n", grads.get(k))
    except NotImplementedError:
        print("Implement rnn_forward and rnn_backward first.")


if __name__ == "__main__":
    main()

Q1. Output sequence y(t):
 [0.3055 1.8262 1.2641 2.0339 0.8278]

Q2. Output deltas delta_y(t):
 [-0.8908 -0.2713  0.8764  0.9886  2.1996]

Q2. Hidden deltas delta_h(t) (column t-1 = time t):
 [[-0.036  -0.0127  0.0337  0.0769  0.0122]
 [ 0.0054  0.0028 -0.0533 -0.     -0.0517]
 [-0.     -0.      0.0005  0.0001  0.004 ]
 [ 0.0002  0.0187 -0.0006 -0.     -0.2748]]

Q3. Gradients:

grad_V:
 [[1.4876 2.9366 2.9118 0.7035]]

grad_U:
 [[ 0.1685  0.1192  0.0224 -1.7854]
 [-0.7974 -0.0631 -0.0199  1.4498]
 [ 0.6344  0.2271 -0.0235  1.6199]]

grad_W:
 [[ 0.1169 -0.0984  0.0043 -0.256 ]
 [ 0.0578 -0.104   0.0045 -0.2752]
 [ 0.0459 -0.105   0.0045 -0.2754]
 [ 0.0787 -0.0205  0.0003  0.0184]]

grad_b:
 [ 0.074  -0.0968  0.0045 -0.2565]

grad_c:
 [2.9025]
